# Rol 4 — El Estratega: dinámica poblacional del AG
Ciclo completo (`src/rol4_ag.py`) y experimentos (`src/rol4_experimentos.py`).
Se estudia **tamaño de población, elitismo, número de generaciones y mutación**, y se mide la convergencia
(máx, media, desviación, diversidad, evaluaciones). Los operadores son los del Rol 3 (combinación C3).

In [1]:
import sys, os, importlib
sys.path.insert(0, os.path.abspath(".."))
os.chdir(os.path.abspath(".."))          # raíz del repo
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from src import rol3_operadores as op, rol4_ag as rol4, rol4_experimentos as e4, rol5_analisis as an

def _cargar(nombre):
    try:    return importlib.import_module(f"src.{nombre}"), True
    except ModuleNotFoundError: return importlib.import_module("src._provisional"), False
rol1, ok1 = _cargar("rol1_codificacion"); rol2, ok2 = _cargar("rol2_fitness")
if not (ok1 and ok2):
    print("⚠️  FITNESS/CODIFICACIÓN PROVISIONALES (Roles 1 y 2 aún no subidos). Repetir con Restart & Run All cuando estén.")
GRAF, TAB = "results/graficos", "results/tablas"
os.makedirs(GRAF, exist_ok=True); os.makedirs(TAB, exist_ok=True)
SEMILLAS = range(5)
base = {"poblacion": 30, "n_genes": rol1.N_GENES, "generaciones": 50, "elitismo": 2,
        "operadores": op.COMBINACIONES["C3 Balance (torneo + blend α=0.5 + no-uniforme)"]}
print("presupuesto base =", base["poblacion"] + base["generaciones"]*(base["poblacion"]-base["elitismo"]), "evaluaciones")

presupuesto base = 1430 evaluaciones


## 1. Ciclo del AG (resumen)
Por generación: ordenar → **elites** pasan intactos → `reproducir` crea `poblacion − elitismo` hijos →
se evalúan solo los hijos → se registra max/media/std/diversidad/evaluaciones → ¿parada?
Evaluaciones totales = `N + T·(N − elitismo)`.

## 2. Configuraciones (cada una cambia un solo parámetro) y corridas

In [2]:
cfgs = e4.configuraciones(base)
df, hist, mejores = e4.correr_configuraciones(rol4.ejecutar_ag, rol2.fitness, cfgs, seeds=SEMILLAS)
tabla = e4.tabla_resumen(df)
tabla.to_csv(f"{TAB}/rol4_configuraciones.csv"); df.to_csv(f"{TAB}/rol4_corridas.csv", index=False)
tabla

,poblacion,elitismo,generaciones,evaluaciones,fit_final,desv,fit_400ev,div_final,gen_colapso,% prematura,Δ vs base
config,,,,,,,,,,,
base,30,2,50.0,1430.0,0.9051,0.0040,0.8984,0.0146,40.4,20.0,0.0000
pob=10,10,2,50.0,410.0,0.9022,0.0042,0.9022,0.0028,27.6,20.0,-0.0029
pob=20,20,2,50.0,920.0,0.9046,0.0050,0.9000,0.0070,37.0,20.0,-0.0005
pob=60,60,2,50.0,2960.0,0.9089,0.0007,0.8906,0.0229,43.8,0.0,0.0038
elit=0,30,0,50.0,1530.0,0.9087,0.0004,0.8993,0.0329,45.8,0.0,0.0037
elit=1,30,1,50.0,1480.0,0.9063,0.0031,0.8961,0.0128,41.0,0.0,0.0013
elit=5,30,5,50.0,1280.0,0.9081,0.0009,0.9012,0.0123,40.0,0.0,0.0030
elit=10,30,10,50.0,1030.0,0.9077,0.0008,0.9016,0.0146,40.0,0.0,0.0027
gen=25,30,2,25.0,730.0,0.9033,0.0021,0.8983,0.0858,22.0,0.0,-0.0018


**Cómo leer la tabla:** `fit_final` = mejor fitness medio de 5 semillas; `fit_400ev` = mejor fitness usando
solo ~400 evaluaciones (mismo costo para todos); `div_final` = diversidad al terminar; `gen_colapso` = primera
generación con diversidad < 0.05; `% prematura` = corridas con diversidad < 0.05 y fitness lejos del mejor conocido.

## 3. Curvas de convergencia

In [3]:
e4.graficar_convergencia(hist, ["base", "pob=10", "pob=60"], GRAF, "rol4_conv_poblacion.png")
e4.graficar_convergencia(hist, ["elit=0", "elit=2", "elit=10"] if "elit=2" in hist else ["elit=0", "base", "elit=10"], GRAF, "rol4_conv_elitismo.png")
e4.graficar_convergencia(hist, ["gen=25", "base", "gen=100"], GRAF, "rol4_conv_generaciones.png")
e4.graficar_vs_evaluaciones(hist, ["pob=10", "base", "pob=60"], GRAF, "rol4_vs_evaluaciones.png")
plt.show()

In [4]:
e4.graficar_todas(hist, GRAF, 'rol4_convergencia_todas.png'); plt.show()

**Caso provocado (`prem:`)**: población 8 y mutación 0.01 para *forzar* la convergencia prematura y probar remedios
(inmigrantes, más mutación, más población).

In [5]:
sel = ["base", "prem: pob=8 pm=0.01", "prem + inmigrantes", "prem + pm=0.15", "prem + pob=30"]
display(tabla.loc[sel])
e4.graficar_convergencia(hist, sel, GRAF, "rol4_prematura_remedios.png")
plt.show()

,poblacion,elitismo,generaciones,evaluaciones,fit_final,desv,fit_400ev,div_final,gen_colapso,% prematura,Δ vs base
config,,,,,,,,,,,
base,30,2,50.0,1430.0,0.9051,0.0040,0.8984,0.0146,40.4,20.0,0.0000
prem: pob=8 pm=0.01,8,2,50.0,308.0,0.8835,0.0053,0.8835,0.0002,8.6,100.0,-0.0216
prem + inmigrantes,8,2,50.0,371.0,0.8890,0.0048,0.8890,0.5688,NaN,0.0,-0.0161
prem + pm=0.15,8,2,50.0,308.0,0.8993,0.0046,0.8993,0.0015,37.0,40.0,-0.0058
prem + pob=30,30,2,50.0,1430.0,0.9052,0.0038,0.8979,0.0045,25.6,0.0,0.0001


## 4. ¿Qué hiperparámetro importa más? (dos lecturas)
Cambiar la población cambia también el **costo**. Por eso se mira (a) con las mismas generaciones y
(b) con el mismo presupuesto de evaluaciones.

In [6]:
a, b, c = e4.efecto_hiperparametros(df)
print("(a) mismas generaciones"); display(a)
print("(b) mismo presupuesto (~400 evaluaciones)"); display(b)
print("(c) población a IGUAL presupuesto total (10 vs 60, ~1430 evaluaciones)"); display(c)
an.graficar_tornado(a, "Rol 4 — efecto con mismas generaciones", GRAF, "rol4_tornado_generaciones.png")
an.graficar_tornado(b, "Rol 4 — efecto con mismo presupuesto (~400 ev.)", GRAF, "rol4_tornado_presupuesto.png")
plt.show()

(a) mismas generaciones


,factor,valor_bajo,valor_alto,fitness_bajo,fitness_alto,base,rango
0,poblacion,10,60,0.902191,0.908869,0.90507,0.006678
1,generaciones,25,100,0.903289,0.907314,0.90507,0.004024
2,elitismo,0,10,0.908723,0.907731,0.90507,0.000992
3,mutación pm,0.02,0.3,0.907798,0.907430,0.90507,0.000368


(b) mismo presupuesto (~400 evaluaciones)


,factor,valor_bajo,valor_alto,fitness_bajo,fitness_alto,base,rango
0,poblacion,10,60,0.902153,0.890626,0.898404,0.011528
1,elitismo,0,10,0.899311,0.901562,0.898404,0.002251
2,mutación pm,0.02,0.3,0.900922,0.899294,0.898404,0.001628


(c) población a IGUAL presupuesto total (10 vs 60, ~1430 evaluaciones)


,factor,valor_bajo,valor_alto,fitness_bajo,fitness_alto,base,rango
0,poblacion (=presup.),10,60,0.907196,0.906928,0.90507,0.000268


## 5. Convergencia prematura y deriva genética

In [7]:
print("Corridas con convergencia prematura:", int(df["prematura"].sum()), "de", len(df))
print("Configuraciones con diversidad final más baja:")
display(tabla["div_final"].sort_values().head(3).to_frame())
e4.graficar_estancamiento(hist, "pob=10", GRAF, "rol4_estancamiento_pob10.png"); plt.show()
nombres = [g[0] if isinstance(g, (tuple, list)) else str(g) for g in getattr(rol1, "GENES", range(rol1.N_GENES))]
deriva = e4.deriva_por_gen(mejores, nombres, "base"); deriva.to_csv(f"{TAB}/rol4_deriva_genes.csv", index=False)
deriva

Corridas con convergencia prematura: 10 de 100
Configuraciones con diversidad final más baja:


,div_final
config,
prem: pob=8 pm=0.01,0.0002
pob=10 (=presup.),0.0005
prem + pm=0.15,0.0015


,gen,desv_entre_semillas,media
0,ataque,0.002,0.002
1,amp_h7,0.011,0.006
2,sostenido,0.021,0.048
3,amp_h5,0.025,0.092
4,amp_h6,0.029,0.098
5,amp_h1,0.063,0.920
6,amp_h2,0.068,0.784
7,amp_h3,0.116,0.721
8,amp_h8,0.169,0.746
9,amp_h4,0.224,0.376


**Lectura:** un gen con desviación entre semillas ≈ 0.29 (la de un uniforme en [0,1]) es casi neutral:
cada corrida lo fija en un valor al azar (deriva). Un gen con desviación baja es importante: todas las corridas
convergen al mismo valor.

In [8]:
ejemplo = rol4.ejecutar_ag({**base, "paciencia": 10, "generaciones": 200}, rol2.fitness, 0)
print("Parada:", ejemplo["motivo_parada"], "en la generación", ejemplo["gen_final"],
      "| evaluaciones:", ejemplo["historial"]["evaluaciones"][-1], "| mejor fitness:", round(ejemplo["mejor_fit"], 4))

Parada: estancamiento en la generación 55 | evaluaciones: 1570 | mejor fitness: 0.9094
